In [1]:
import numpy as np

# Carga de datos
data = np.load('data.npy')
x_all = data[:, 0]
y_all = data[:, 1]

# Tomar 4 puntos alrededor del centro del transito (t ~ 0)
x_datos = x_all[96:100]
y_datos = y_all[96:100]

# Definir x_eval justo a mitad de camino entre los dos puntos centrales
x_eval = (x_datos[1] + x_datos[2]) / 2.0

# Comprobacion en pantalla
print("x_datos:", x_datos)
print("x_eval:", x_eval)
print("¿x_eval esta dentro del rango?:", x_datos.min() <= x_eval <= x_datos.max())

x_datos: [-0.00111111 -0.00018519  0.00074074  0.00166667]
x_eval: 0.0002777777777779114
¿x_eval esta dentro del rango?: True


# Justificación de la selección de datos e interpolación

En un inicio se intentó evaluar directamente el valor de `x_eval = -0.25` utilizando los 4 puntos centrales del tránsito ($t \approx 0$). Sin embargo, al comprobar el rango con `x_datos.min() <= x_eval <= x_datos.max()`, la condición dio como resultado `False`, lo que produjo un valor de flujo absurdo de aproximadamente `-85.20`. Esto ocurrió porque evaluar en $x = -0.25$ fuera del dominio de esos datos implicaba realizar una **extrapolación polinómica**, provocando una divergencia numérica drástica.

Dado que la intención del ejercicio planteado en la diapositiva es estimar el flujo *"a mitad de camino entre dos muestras cercanas al mínimo del tránsito"*, se ajustó la selección de datos a los 4 puntos que abarcan el fondo del tránsito ($x \in [-0.0011, 0.0016]$) y se definió $x_{eval}$ exactamente en el punto medio entre las dos muestras centrales inmediatas al mínimo ($x \approx -0.000185$). De esta forma, $x_{eval}$ queda estrictamente dentro del intervalo (`True`), garantizando una **interpolación directa** numéricamente estable y físicamente coherente.

# Evaluacion y despliegue del resultado
flujo_estimado = interp_directo(x_datos, y_datos, x_eval)
print(flujo_estimado)

In [2]:
# Construccion de la Matriz de Vandermonde
def vandermonde(x_datos):
    n = len(x_datos)
    V = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            V[i, j] = x_datos[i] ** j
    return V

In [3]:
# Eliminacion Gaussiana con Pivoteo Parcial
def gauss_solve(A, b):
    n = len(b)
    M = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])
    
    for k in range(n):
        piv = k + np.argmax(np.abs(M[k:n, k]))
        M[[k, piv]] = M[[piv, k]]
        
        for i in range(k + 1, n):
            factor = M[i, k] / M[k, k]
            for j in range(k, n + 1):
                M[i, j] = M[i, j] - factor * M[k, j]
                
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        suma = 0.0
        for j in range(i + 1, n):
            suma = suma + M[i, j] * x[j]
        x[i] = (M[i, n] - suma) / M[i, i]
        
    return x

In [4]:
# Interpolacion Directa
def interp_directo(x_datos, y_datos, x_eval):
    V = vandermonde(x_datos)
    a = gauss_solve(V, y_datos)
    
    resultado = 0.0
    for j in range(len(a)):
        resultado = resultado + a[j] * (x_eval ** j)
        
    return resultado

In [ ]:
# Evaluacion y resultado
flujo_estimado = interp_directo(x_datos, y_datos, x_eval)
print(flujo_estimado)

0.9822516088452504
